# Step 13.1 — Why Multiple Heads：一个 Causal Attention Head 为什么还不够？

Step 12 已经完成一个完整 single causal self-attention head：

$$
O
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_h}}+M
\right)V.
$$

一个 head 已经能：

- 用 learned Q/K 判断哪些合法 prefix positions 更相关；
- 用 V 携带 payload；
- 用 causal mask 禁止 future leakage。

那为什么 Transformer 还要使用多个 heads？

核心动机不是“一个 head 完全不会工作”，而是：

> **同一个 token representation 可能有多种有用的匹配视角；多套独立 learned projections 可以让模型在多个 learned subspaces 中并行建立不同的关系模式。**

本节只解释这个动机和最小数学直觉，不进入 Step 13.2 的 fused QKV projection，也不实现完整 Multi-Head Attention。


## 1. 一个 Head 本质上只有一套 Q/K/V View

假设输入：

$$
X\in\mathbb R^{B\times T\times C}.
$$

一个 single head 拥有：

$$
W_Q,\quad W_K,\quad W_V.
$$

于是：

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

也就是说，这个 head 对所有 positions 使用同一套 learned feature spaces：

```text
Query space
Key space
Value space
```

这种设计已经很强，但它仍然只有**一套同时负责当前所有 interaction 的投影视角**。

如果输入 representation 中同时包含很多不同类型、不同组合方式的 feature，一个 head 必须把这些需求都压进同一套 Q/K/V subspace 中。

这并不是数学上“不可能”，而是表示能力会更受约束。


## 2. Multiple Heads：给模型多套独立 Learned Subspaces

现在假设有两个 heads：

$$
h=1,2.
$$

每个 head 都可以拥有自己独立的 projection parameters：

$$
W_Q^{(h)},
\quad
W_K^{(h)},
\quad
W_V^{(h)}.
$$

于是同一个 X 可以产生：

$$
Q^{(h)}=XW_Q^{(h)},
\qquad
K^{(h)}=XW_K^{(h)},
\qquad
V^{(h)}=XW_V^{(h)}.
$$

每个 head 再独立形成自己的 causal attention：

$$
A^{(h)}
=
\operatorname{softmax}
\left(
\frac{Q^{(h)}K^{(h)T}}{\sqrt{d_h}}+M
\right).
$$

因此：

```text
same X
├─ Head 1 learned projection space
│  └─ its own attention pattern
│
└─ Head 2 learned projection space
   └─ its own attention pattern
```

这就是 **multiple learned subspaces（多个可学习子空间）** 的核心。

这里的“subspace”主要表达不同 learned linear projections 形成的不同 feature views，不要求把每个 head 理解成某个固定的人类概念模块。


## 3. 不要预设：Head 1 = Syntax，Head 2 = Pronoun

一个常见但危险的教学捷径是：

```text
Head 1 专门学语法
Head 2 专门学指代
Head 3 专门学位置
```

这种说法不应该作为 architecture definition。

真实模型中，head 的 learned behavior 取决于：

- 初始化；
- 数据；
- optimization；
- layer；
- 当前输入；
- 与其他 heads / layers 的协同。

一个 head 可能表现出某种可观察 pattern，也可能：

- 同时参与多种关系；
- 与其他 heads 部分冗余；
- 在不同输入上改变关注模式；
- 很难对应一个干净的人类标签。

所以更准确的说法是：

> **Multiple heads 提供多个独立可学习的 interaction subspaces；模型可以利用它们形成不同模式，但我们不能在训练前给每个 head 固定人类语义。**


## 4. Tiny Example：同一个 X，两套 Head Projections 得到不同 Attention Pattern

为了把“不同 learned subspaces”变成可手算的东西，先暂时去掉 Batch，取：

$$
X
=
\begin{bmatrix}
1&0\\
0&1\\
1&1
\end{bmatrix}
\in
\mathbb R^{3\times2}.
$$

也就是：

```text
T = 3
C = 2
d_h = 1
```

我们手工构造两个 heads。

### Head A：只取第 1 个 input feature

$$
W_Q^{(A)}
=
W_K^{(A)}
=
\begin{bmatrix}
1\\
0
\end{bmatrix}.
$$

所以：

$$
Q^{(A)}
=
K^{(A)}
=
\begin{bmatrix}
1\\
0\\
1
\end{bmatrix}.
$$

### Head B：只取第 2 个 input feature

$$
W_Q^{(B)}
=
W_K^{(B)}
=
\begin{bmatrix}
0\\
1
\end{bmatrix}.
$$

得到：

$$
Q^{(B)}
=
K^{(B)}
=
\begin{bmatrix}
0\\
1\\
1
\end{bmatrix}.
$$

注意：这里的 projection matrices 是**人为设计的教学例子**。真实模型中的 head projections 是训练得到的，不会被规定成“第一个 feature head / 第二个 feature head”。


## 5. 两个 Heads 的 Causal Attention 为什么不同？

因为 `d_h=1`，scaling factor 是 1。

### Head A

未 mask scores：

$$
Q^{(A)}K^{(A)T}
=
\begin{bmatrix}
1&0&1\\
0&0&0\\
1&0&1
\end{bmatrix}.
$$

加入 causal mask 后，softmax 大约得到：

$$
A^{(A)}
\approx
\begin{bmatrix}
1&0&0\\
0.500&0.500&0\\
0.422&0.155&0.422
\end{bmatrix}.
$$

### Head B

未 mask scores：

$$
Q^{(B)}K^{(B)T}
=
\begin{bmatrix}
0&0&0\\
0&1&1\\
0&1&1
\end{bmatrix}.
$$

causal softmax 后：

$$
A^{(B)}
\approx
\begin{bmatrix}
1&0&0\\
0.269&0.731&0\\
0.155&0.422&0.422
\end{bmatrix}.
$$

同一个 sequence representations X，因为两个 heads 使用不同 projection spaces，就得到了不同的 pairwise compatibility 与 attention pattern。

这才是 multiple heads 最重要的直觉：

> **不是复制同一个 attention 多次，而是给模型多套独立参数，让不同 heads 有能力学习不同的 interaction views。**


In [ ]:
import mlx.core as mx

X = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])  # (T,C) = (3,2)

# Hand-designed projections for intuition only.
W_A = mx.array([[1.0], [0.0]])  # (C,d_h)
W_B = mx.array([[0.0], [1.0]])  # (C,d_h)


def causal_weights_from_projection(x, w_q, w_k):
    q = x @ w_q                    # (T,d_h)
    k = x @ w_k                    # (T,d_h)
    scores = q @ k.T               # (T,T), d_h=1 so scale=1

    T = x.shape[0]
    future = mx.triu(mx.ones((T, T)), k=1) > 0
    mask = mx.where(future, float('-inf'), 0.0)

    masked_scores = scores + mask
    weights = mx.softmax(masked_scores, axis=-1)
    return q, k, scores, weights


q_a, k_a, scores_a, A_a = causal_weights_from_projection(X, W_A, W_A)
q_b, k_b, scores_b, A_b = causal_weights_from_projection(X, W_B, W_B)

print('Head A Q:', q_a)
print('Head A scores:')
print(scores_a)
print('Head A attention:')
print(A_a)

print('Head B Q:', q_b)
print('Head B scores:')
print(scores_b)
print('Head B attention:')
print(A_b)

# This master notebook is unexecuted.
# The matrices above are hand-designed, so their expected patterns can be
# checked analytically; they are not claims about learned semantic head roles.


## 6. Multiple Heads 真正增加的是什么？

一个 single head 已经能形成一个 context-dependent representation。

多个 heads 增加的是：

### A. 多套 Matching Geometry

不同：

$$
W_Q^{(h)},W_K^{(h)}
$$

可以让“什么叫匹配”在不同 heads 中有不同 learned geometry。

### B. 多套 Payload Views

不同：

$$
W_V^{(h)}
$$

意味着即使两个 heads 暂时关注相似 positions，它们取回的 Value representation 也可以不同。

### C. 减少单一 Head 的表示压力

不再要求一套 Q/K/V projections 同时承担所有 useful interaction patterns。

但是这些只是能力上的增加，不是保证：

```text
更多 heads
= 一定更多有意义的人类可解释功能
```

也不能推导出每个 head 必然互不重复。


## 7. Step 13.1 材料边界

- single head 只有一套 Q/K/V learned projection spaces；multiple heads 的核心动机是提供多套独立可学习 interaction subspaces。
- 对 head `h`，可以概念性写成 `Q^(h)=XW_Q^(h)`、`K^(h)=XW_K^(h)`、`V^(h)=XW_V^(h)`，每个 head 可形成自己的 causal attention pattern。
- 多 heads 增加 matching views 与 payload views 的表示灵活性，但不保证 heads 一定学到互斥或人类可解释的功能。
- 不应在 architecture 层面预设“某个 head = syntax / pronoun / position”等固定语义。
- tiny two-head example 只用手工 projection matrices 展示不同 subspaces 可以导致不同 attention pattern；它不是完整 Multi-Head Attention 实现。
- 本轮只准备 Step 13.1；不进入 Step 13.2，不介绍 fused `Linear(C,3C)`、QKV split、channel-to-head reshape、head concatenation 或 output projection。
